# Advection — advection of a magnetic bump

`MagneticStandalone` · ASN 28 validation case (TRUST-LiMe).


## Introduction

Validation of the **standalone magnetic (induction) equation** added by
TRUST-LiMe:

$$\partial_t b + (u \cdot \nabla) b - \nabla \cdot (\eta \nabla b) = 0$$

with the magnetic diffusivity $\eta$ and the transporting velocity $u$ supplied
as data (no Navier–Stokes coupling in this first phase). Every case is
**self-contained** (inline `Mailler`/`Pave` mesh, no `.med` file) and records
point probes (`Postraitement/Sondes`) into `.son` files which are compared
below against a closed-form solution.

This notebook runs **one and only one test case**: ``advection``. Per the TRUST
validation-form layout (`Validation/Rapports_automatiques/<Category>/<FormName>/`),
the other `MagneticStandalone` cases live in their own sibling folders, each
with `<FormName>/<FormName>.ipynb` + `src/<case>.data`.

## What it validates

A Gaussian bump advected by a constant $u=(0.2,0)$. The upstream scheme is
diffusive so the bump broadens and its peak is lowered by numerical diffusion;
the transported bump must still arrive at the advected location $x_0+Ut$ with
a substantial fraction of its initial amplitude.

## Contracts checked

| # | Modelling decision | Check |
|---|---|:--:|
| 7 | velocity is given data, frozen | bump transported to the right |
| — | convection operator (`vitesse_magnetique` reading path) | bump transported; arrival peak ≥ 50% of initial |


In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()


## Helpers: read a TRUST probe (`.son`) file

The case is copied by `trustutils` into `run.BUILD_DIRECTORY`; the `.son`
probe files are written next to the dataset in `src/` and copied over in the
run cell below, where `read_son()` expects them.


In [ ]:
import numpy as np
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY

def read_son(case, name):
    """Return (time, values) for probe `name` of case `case` (dataset name)."""
    path = f"{BUILD}/{case}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def ratio_error(case, tags, ref):
    """Max |b(t)/b(0) - ref(t)| over the given probes and all recorded times."""
    worst = 0.0
    for tag in tags:
        t, b = read_son(case, tag)
        if b[0] == 0:
            continue
        worst = max(worst, float(np.max(np.abs(b / b[0] - ref(t)))))
    return worst

print("BUILD =", BUILD)


## Run

Runs on **2 processors** (`nbProcs=2` + `case.partition()`). The dataset's
commented `# BEGIN PARTITION` / `# BEGIN SCATTER` blocks let `trust -partition`
build the parallel `PAR_` variant (upstream `grad_rot.data` pattern).


In [ ]:
import pathlib, shutil

from trustutils import run

NB_PROCS = 2

run.reset()
case = run.addCase(".", "advection.data", nbProcs=NB_PROCS)
case.partition()          # trust -partition advection.data 2 -> PAR_advection.data + DOM_000*.Zones
run.printCases()
run.runCases()

# trustutils runs the case in its build tree; the .son probes are written next
# to the dataset in src/, so copy them where read_son() looks for them.
build = pathlib.Path(run.BUILD_DIRECTORY)
build.mkdir(parents=True, exist_ok=True)
for son in pathlib.Path("src").glob(f"{case.dataFileName_}_*.son"):
    shutil.copy2(son, build / son.name)


## Checks


In [ ]:
t, b_peak0 = read_son(case.dataFileName_, "SONDE_X025")  # initial peak location x0
t2, b_trav  = read_son(case.dataFileName_, "SONDE_X045")  # advected location x0+Ut
peak0 = b_peak0[0]
print(f"initial peak b(x0,0)          : {peak0:.4f}")
print(f"b at x0 at final time         : {b_peak0[-1]:.4f}   (bump has left)")
print(f"b at x0+Ut : {b_trav[0]:.4f} -> {b_trav[-1]:.4f}")

# transport: the bump must leave the initial location ...
assert b_peak0[-1] < 0.3, f"bump did not leave x0: {b_peak0[-1]:.3f}"
# ... and arrive at x0+Ut with a substantial fraction of the initial peak.
# amont upwind broadens the Gaussian (numerical diffusion), so the arrival
# peak is lowered; keep the bound honest (>= 50% of the initial peak).
assert b_trav[0] < 0.3, f"bump present at x0+Ut at t=0: {b_trav[0]:.3f}"
ratio = b_trav[-1] / peak0
print(f"arrival peak / initial peak   : {ratio:.3f}")
assert ratio > 0.5, f"peak not transported: arrival {b_trav[-1]:.3f} < 0.5*{peak0:.3f}"
print("OK: bump advected to the right; peak preserved within amont diffusion.")

import matplotlib.pyplot as plt
plt.figure(figsize=(6,4))
plt.plot(t, b_peak0, label="b(x0,t)")
plt.plot(t2, b_trav, label="b(x0+Ut,t)")
plt.xlabel("t [s]"); plt.ylabel("b"); plt.legend(); plt.grid(True); plt.show()


## Computer Performance


In [ ]:
run.tablePerf()